In [2]:
import os
import time
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

client = OpenAI()

print("Environment setup successful!")

Environment setup successful!


In [3]:
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "user", "content": "Reply with exactly: API connection successful"}
    ],
    temperature=0
)

print(response.choices[0].message.content)

API connection successful


In [5]:

# PART 1 — Initial prompts and baseline test

import time

def call_openai(prompt):
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )
    return response.choices[0].message.content

sentiment_prompt_v1 = """
Classify this customer message:
"I love this product! It's exactly what I needed."
"""

product_prompt_v1 = """
Create a product description for a wireless mouse that costs $29.99.
"""

extraction_prompt_v1 = """
Extract information from this customer feedback:
"I ordered item #12345 on March 15th.
The delivery was fast but the packaging was damaged."
"""

prompts_v1 = {
    "Sentiment Analysis": sentiment_prompt_v1,
    "Product Description": product_prompt_v1,
    "Data Extraction": extraction_prompt_v1
}

baseline_results = {}

for task, prompt in prompts_v1.items():
    print(f"\n--- {task} ---")
    result = call_openai(prompt)
    baseline_results[task] = result
    print(result)



--- Sentiment Analysis ---
This customer message can be classified as **Positive Feedback** or **Customer Satisfaction**.

--- Product Description ---
**Product Name: SwiftClick Wireless Mouse**

**Price: $29.99**

**Product Description:**

Elevate your computing experience with the SwiftClick Wireless Mouse, the perfect blend of style, comfort, and functionality. Designed for both home and office use, this sleek and modern mouse offers a seamless connection and exceptional performance, all at an unbeatable price.

**Key Features:**

- **Wireless Freedom:** Say goodbye to tangled cords! The SwiftClick connects effortlessly via a reliable 2.4GHz wireless connection, providing you with the freedom to move around your workspace without restrictions.

- **Ergonomic Design:** Crafted with your comfort in mind, the SwiftClick features an ergonomic shape that fits perfectly in your hand, reducing strain during long hours of use. Its textured grip ensures maximum control and precision.

- **H

In [6]:

# PART 2 — Repeated testing: 5, 10 and 15 runs

repeated_results = {}

for task, prompt in prompts_v1.items():
    repeated_results[task] = {}

    for n in [5, 10, 15]:
        results = []

        print(f"\nTesting {task}: {n} runs")

        for i in range(n):
            result = call_openai(prompt)
            results.append(result)
            time.sleep(1)

        repeated_results[task][n] = results

        print(f"Completed {n} runs.")
        print("Unique responses:", len(set(results)))



Testing Sentiment Analysis: 5 runs
Completed 5 runs.
Unique responses: 1

Testing Sentiment Analysis: 10 runs
Completed 10 runs.
Unique responses: 1

Testing Sentiment Analysis: 15 runs
Completed 15 runs.
Unique responses: 1

Testing Product Description: 5 runs
Completed 5 runs.
Unique responses: 4

Testing Product Description: 10 runs
Completed 10 runs.
Unique responses: 10

Testing Product Description: 15 runs
Completed 15 runs.
Unique responses: 14

Testing Data Extraction: 5 runs
Completed 5 runs.
Unique responses: 1

Testing Data Extraction: 10 runs
Completed 10 runs.
Unique responses: 1

Testing Data Extraction: 15 runs
Completed 15 runs.
Unique responses: 1


In [7]:

# Analyze baseline results

from collections import Counter
import pandas as pd

analysis_rows = []

for task, tests in repeated_results.items():
    for n, outputs in tests.items():
        counts = Counter(outputs)
        most_common_count = counts.most_common(1)[0][1]

        analysis_rows.append({
            "Task": task,
            "Runs": n,
            "Unique responses": len(counts),
            "Exact-match consistency (%)": round(
                most_common_count / n * 100, 1
            )
        })

baseline_analysis = pd.DataFrame(analysis_rows)
display(baseline_analysis)


,Task,Runs,Unique responses,Exact-match consistency (%)
0,Sentiment Analysis,5,1,100.0
1,Sentiment Analysis,10,1,100.0
2,Sentiment Analysis,15,1,100.0
3,Product Description,5,4,40.0
4,Product Description,10,10,10.0
5,Product Description,15,14,13.3
6,Data Extraction,5,1,100.0
7,Data Extraction,10,1,100.0
8,Data Extraction,15,1,100.0


In [8]:

# Part 3 — Iteration 1: clearer prompts

sentiment_prompt_v2 = """
Classify the customer message as Positive, Negative, or Neutral.
Return exactly one word and nothing else.

Message: "I love this product! It's exactly what I needed."
"""

product_prompt_v2 = """
Write a product description for a wireless mouse priced at $29.99.

Requirements:
- Use 3 sentences.
- Use a professional, appealing tone.
- Mention the product type and price.
- Do not invent technical specifications or features.
"""

extraction_prompt_v2 = """
Extract the following fields from the customer feedback:
- item_id
- order_date
- delivery_feedback
- packaging_condition

Return valid JSON only. Use null for information that is not provided.
Do not invent missing information.

Feedback: "I ordered item #12345 on March 15th.
The delivery was fast but the packaging was damaged."
"""

prompts_v2 = {
    "Sentiment Analysis": sentiment_prompt_v2,
    "Product Description": product_prompt_v2,
    "Data Extraction": extraction_prompt_v2
}

iteration1_results = {}

for task, prompt in prompts_v2.items():
    outputs = []

    for i in range(5):
        outputs.append(call_openai(prompt))
        time.sleep(1)

    iteration1_results[task] = outputs

    print(f"\n--- {task}: Iteration 1 ---")
    for i, output in enumerate(outputs, start=1):
        print(f"{i}. {output}")



--- Sentiment Analysis: Iteration 1 ---
1. Positive
2. Positive
3. Positive
4. Positive
5. Positive

--- Product Description: Iteration 1 ---
1. Experience seamless navigation with our sleek wireless mouse, available for just $29.99. Designed for both comfort and efficiency, this mouse enhances your productivity while eliminating the hassle of tangled cords. Elevate your workspace with this stylish and reliable accessory that perfectly balances form and function.
2. Experience seamless navigation with our sleek wireless mouse, available for just $29.99. Designed for both comfort and efficiency, this mouse enhances your productivity while eliminating the hassle of tangled cords. Elevate your workspace with this stylish and reliable accessory that perfectly balances form and function.
3. Experience seamless navigation with our sleek wireless mouse, available for just $29.99. Designed for both comfort and efficiency, this mouse enhances your productivity while eliminating the hassle of t

In [14]:

# Compare baseline, Iteration 1 and Iteration 2

tasks = [
    "Sentiment Analysis",
    "Product Description",
    "Data Extraction"
]

for task in tasks:
    print(f"\n--- {task} ---")

    versions = [
        ("Baseline", repeated_results[task][5]),
        ("Iteration 1", iteration1_results[task]),
        ("Iteration 2", iteration2_results[task])
    ]

    for name, outputs in versions:
        unique = len(set(outputs))
        total = len(outputs)
        print(f"{name}: {unique} unique responses out of {total}")



--- Sentiment Analysis ---


NameError: name 'iteration2_results' is not defined

In [15]:

# Create and run Iteration 2 prompts

sentiment_prompt_v3 = """
Classify the message as Positive, Negative, or Neutral.
Examples:
"I love it!" -> Positive
"I hate it!" -> Negative
"It arrived today." -> Neutral

Return only one label.

Message: "I love this product! It's exactly what I needed."
"""

product_prompt_v3 = """
Create a wireless mouse product description priced at $29.99.

Use exactly this format:
Headline: one short line
Description: exactly two sentences
Price: $29.99

Use an appealing professional tone.
Do not invent product features or specifications.
"""

extraction_prompt_v3 = """
Extract the feedback into valid JSON only, without Markdown fences.

Use these keys:
item_id, order_date, delivery_feedback, packaging_condition

Use null for missing information. Do not invent details.

Example:
Feedback: "Item #77 arrived late."
Output: {"item_id":"77","order_date":null,"delivery_feedback":"arrived late","packaging_condition":null}

Feedback: "I ordered item #12345 on March 15th. The delivery was fast but the packaging was damaged."
"""

prompts_v3 = {
    "Sentiment Analysis": sentiment_prompt_v3,
    "Product Description": product_prompt_v3,
    "Data Extraction": extraction_prompt_v3
}

iteration2_results = {}

for task, prompt in prompts_v3.items():
    outputs = []

    for i in range(5):
        outputs.append(call_openai(prompt))
        time.sleep(1)

    iteration2_results[task] = outputs
    print(f"{task}: {len(set(outputs))} unique responses out of 5")


Sentiment Analysis: 1 unique responses out of 5
Product Description: 2 unique responses out of 5
Data Extraction: 1 unique responses out of 5


In [16]:

# Part 5 — Test improved prompts with new inputs

new_tests = {
    "Sentiment Analysis": (
        sentiment_prompt_v3,
        "The delivery was late and the item arrived broken."
    ),
    "Product Description": (
        product_prompt_v3,
        "Create a product description for a ceramic coffee mug priced at $12.99."
    ),
    "Data Extraction": (
        extraction_prompt_v3,
        'I purchased item #98765 on April 2nd. It arrived on time, but the box was crushed.'
    )
}

for task, (prompt, new_input) in new_tests.items():
    # Replace the original example with a new input
    if task == "Sentiment Analysis":
        test_prompt = prompt.replace(
            "I love this product! It's exactly what I needed.",
            new_input
        )
    elif task == "Product Description":
        test_prompt = new_input
    else:
        test_prompt = prompt.replace(
            'I ordered item #12345 on March 15th. The delivery was fast but the packaging was damaged.',
            new_input
        )

    print(f"\n--- {task}: New Input Test ---")
    print(call_openai(test_prompt))



--- Sentiment Analysis: New Input Test ---
Negative

--- Product Description: New Input Test ---
**Product Description: Handcrafted Ceramic Coffee Mug - $12.99**

Elevate your coffee experience with our beautifully handcrafted ceramic coffee mug, designed for both style and functionality. Priced at just $12.99, this mug is the perfect companion for your morning brew, afternoon tea, or cozy hot chocolate.

**Key Features:**

- **Artisan Quality:** Each mug is meticulously crafted by skilled artisans, ensuring that no two pieces are exactly alike. The unique glaze finish adds a touch of elegance, making it a standout addition to your kitchen or office.

- **Generous Capacity:** With a comfortable 12 oz capacity, this mug is perfect for your favorite beverages, whether you prefer a robust espresso or a soothing herbal tea.

- **Ergonomic Design:** The thoughtfully designed handle provides a comfortable grip, allowing you to enjoy your drink without worry. The wide base ensures stability,

### Prompt Engineering Lab — Results and Conclusion

**Objective:** Improve the consistency and reliability of three tasks: sentiment analysis, product description generation, and data extraction.

**Baseline testing:** The initial prompts showed different levels of response consistency. Sentiment analysis produced identical responses, while product descriptions showed substantial variation. Data extraction was evaluated for consistency and output structure.

**Iteration 1:** Clearer instructions and output constraints were introduced. Sentiment analysis was restricted to one label, product descriptions received sentence and content requirements, and data extraction was structured around specified JSON fields.

**Iteration 2:** Examples and stricter formatting instructions were added. The five-run tests produced one unique response for sentiment analysis, two for product description generation, and one for data extraction.

**Testing with new inputs:** Sentiment analysis correctly classified a negative customer message. Data extraction returned the requested information in JSON format. However, product description generation did not follow the required structure and introduced product specifications that were not provided in the input.

**Conclusion:** Clear instructions, examples, and output constraints can improve consistency and structure. However, consistency alone does not guarantee accuracy or compliance. Product description generation requires further refinement and validation to prevent unsupported claims and enforce the required output format.